# Windows | Geospatial UV Python Environment 

This notebook provides a Windows workflow for setting up:

- **Python 3.12** with `uv`
- **Development tools**: ipykernel, ruff
- **Geospatial tools**: GDAL, rasterio, numpy
- **GPU stack**: PyTorch (CUDA 12.x), terratorch, thor-terratorch-ext

Follow each section in order. Execute shell commands in PowerShell from the project directory root. 

## Step 1: Install UV on Windows

UV is a fast Python package manager. Install it using PowerShell:

**Recommendation**
- Use PowerShell with `ExecutionPolicy ByPass` to avoid policy errors.

In [ ]:
# Install UV on Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

**Restart Visual Studio Code and verify the installation**


In [ ]:
# Verify UV installation
where uv 
# Expected output: C:\Users\<username>\.local\bin\uv.exe

uv --version
# Expected output: UV <version_number>

**If you hit Windows policy errors**

Uninstall `uv` and reinstall using the PowerShell command above.

In [ ]:
# Uninstall UV (PowerShell)
powershell -Command "Remove-Item -Force $HOME\.local\bin\uv.exe"

## Step 2: Create New Project with Python 3.12

Create a new directory, initialize a UV project, and set up a Python 3.12 virtual environment.
Replace `<project_dir>` with your preferred project name (for example: `gis-py312-cuda12`).

**Recommendation**
- Keep a consistent folder structure, for example `%USERPROFILE%/git/<org>/<repo>`.

In [ ]:
# Create and navigate to project
# project_dir = %USERPROFILE%/git/github-profile/github-repo

mkdir {project_dir}
cd {project_dir} 

# Create and initialize project
uv init

In [ ]:
# Install Python 3.12 and create virtual environment
uv python install 3.12
cd {project_dir} 
uv venv --python 3.12
.venv\Scripts\Activate.ps1

In [ ]:
# Verify Python 3.12 is active
python --version

## Step 3: Configure Your Project (`pyproject.toml`)

`uv init` creates a minimal project file. Next, add the dependencies and indexes required for geospatial + GPU workflows.

```toml
# Initial pyproject.toml from uv init (minimal template)


# TODO: list structure here
```

#### **Install dependencies for Notebook Development**

The packages `ipykernle` and `pip` are mandatory for selecting the virtual environment as your notebook kernel

In [ ]:
# Install Jupyter kernel and pip as dev dependencies
uv add ipykernel pip --optional dev

Note that your `pyproject.toml` is now updated with these packages inside the [dev] group. 

**Sync dependencies**

In [ ]:
# Sync the pyproject.toml file with the virtual environment (.venv)
uv sync --all-extras

**Verify your environment can be selected as Notebook kernel** 

- Navigate to the Notebook and open in VS Code
- Select the `.venv` as kernel in the upper right corner of the notebook.
- Run the cell below to verify your kernel 

In [1]:
# Check working directory and Python executable
from pathlib import Path
import sys

print(f"Working directory: {Path.cwd()}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

Working directory: c:\Users\wilaca\git\miljodirektoratet\gis-utils-public\notebooks
Python executable: c:\Users\wilaca\git\miljodirektoratet\gis-utils-public\.venv\Scripts\python.exe
Python version: 3.12.13 (main, May 10 2026, 19:35:37) [MSC v.1944 64 bit (AMD64)]


### **Add UV Sources and Indexes for Geospatial and GPU-based packages**

Add the following snippet to the bottom of `pyproject.toml`:

```toml
#  UV package indexes and sources for geospatial, PyTorch, and terratorch
[tool.uv.sources]
gdal = [
  { index = "gdal-wheels", marker = "sys_platform == 'linux'" },
  { index = "geospatial_wheels", marker = "sys_platform == 'win32'" },
]
torch = { index = "pytorch-cu128" }
torchvision = { index = "pytorch-cu128" }
torchaudio = { index = "pytorch-cu128" }
thor-terratorch-ext = { git = "https://github.com/FM4CS/thor_terratorch_ext.git" }

[[tool.uv.index]]
name = "geospatial_wheels"
url = "https://nathanjmcdougall.github.io/geospatial-wheels-index/"
explicit = true

[[tool.uv.index]]
name = "gdal-wheels"
url = "https://gitlab.com/api/v4/projects/61637378/packages/pypi/simple"
explicit = true

[[tool.uv.index]]
name = "pytorch-cu128"
url = "https://download.pytorch.org/whl/cu128"
explicit = true
```

This configuration ensures geospatial and GPU-based packages are installed from the correct indexes.
--all-dev

#### **Install Spatial Stack**

In [ ]:
# Update pyproject.toml with geospatial dependencies and sync with .venv
uv add rasterio gdal numpy
uv sync --all-extras

In [2]:
# Verify GDAL and Rasterio
from osgeo import gdal
import rasterio
import numpy as np

print("GDAL version:", gdal.VersionInfo())
print("GDAL driver count:", gdal.GetDriverCount())
print("Rasterio version:", rasterio.__version__)
print("Rasterio GDAL version:", rasterio.__gdal_version__)
print("NumPy version:", np.__version__)

GDAL version: 3120200
GDAL driver count: 204
Rasterio version: 1.5.0
Rasterio GDAL version: 3.12.1
NumPy version: 2.4.4


#### **Install GPU Stack**

In [ ]:
# Check NVIDIA GPU Hardware
nvidia-smi

In [ ]:
# Update pyproject.toml with GPU dependencies and sync with .venv
uv add torch torchvision torchaudio --optional pytorch

# Install terratorch and extension
# Terratorch extension requires terratorch>=1.2.4
uv add "terratorch==1.2.4" --optional pytorch
uv add "git+https://github.com/FM4CS/thor_terratorch_ext.git" --optional pytorch

uv sync --all-extras

In [5]:
# Verify PyTorch and CUDA
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU count: 2
GPU device name: NVIDIA GeForce RTX 2080 SUPER


In [ ]:
# Verify Terratorch
import terratorch

print("Terratorch version:", terratorch.__version__)

# Verify Thor-Terratorch Extension
import thor_terratorch_ext

print("Thor-Terratorch extension available")

## Step 4: Verify Environment Setup

Connect the notebook to the correct `.venv` kernel created above.

Run the cells below to verify package imports and runtime availability.

In [6]:
# Check working directory and Python executable
from pathlib import Path
import sys

print(f"Working directory: {Path.cwd()}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

Working directory: c:\Users\wilaca\git\miljodirektoratet\gis-utils-public\notebooks
Python executable: c:\Users\wilaca\git\miljodirektoratet\gis-utils-public\.venv\Scripts\python.exe
Python version: 3.12.13 (main, May 10 2026, 19:35:37) [MSC v.1944 64 bit (AMD64)]


In [7]:
# Verify GDAL and Rasterio
from osgeo import gdal
import rasterio
import numpy as np

print("GDAL version:", gdal.VersionInfo())
print("GDAL driver count:", gdal.GetDriverCount())
print("Rasterio version:", rasterio.__version__)
print("Rasterio GDAL version:", rasterio.__gdal_version__)
print("NumPy version:", np.__version__)

GDAL version: 3120200
GDAL driver count: 204
Rasterio version: 1.5.0
Rasterio GDAL version: 3.12.1
NumPy version: 2.4.4


In [8]:
# Verify PyTorch and CUDA
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU count: 2
GPU device name: NVIDIA GeForce RTX 2080 SUPER


In [ ]:
# Verify Terratorch
import terratorch
import thor_terratorch_ext

print("Thor-Terratorch extension available")

In [ ]:
# Check NVIDIA GPU Hardware
!nvidia-smi

## Summary

Environment ready with:

- **Geospatial**: GDAL, rasterio, numpy
- **GPU**: PyTorch (CUDA 12.x), terratorch, thor-terratorch-ext
- **Development**: pytest, ruff, ipykernel

Next steps:
1. Use this environment in notebooks and scripts
2. Run `uv run <command>` for project-scoped execution
3. Build geospatial + deep learning workflows